In [0]:
# ==========================================
# Notebook: 03_gold_kpis
# Project: Pharma Sales Medallion Pipeline
# Ticket: BTSA-003
# Purpose: Create business-ready Gold KPI tables
# ==========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
prescriptions = spark.read.table('silver_prescriptions')
products = spark.read.table('silver_products')
providers = spark.read.table('silver_providers')
regions = spark.read.table('silver_regions')


In [0]:
fact_prescriptions = (
    prescriptions.alias("p")
    .join(products.alias("prod"), "product_id", "left")
    .join(providers.alias("prov"), "provider_id", "left")
    .join(regions.alias("reg"), "region_id", "left")
)

In [0]:
display(fact_prescriptions)

In [0]:
# monthly sales kpi
gold_monthly_sales = (
    fact_prescriptions
    .withColumn("sales_month", F.date_trunc("month", F.col("prescription_date")))
    .groupBy("sales_month")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_net_sales"),
        F.countDistinct("prescription_id").alias("total_prescriptions"),
        F.countDistinct("provider_id").alias("active_providers"),
        F.round(F.avg("discount"), 2).alias("avg_discount")
    )
    .orderBy("sales_month")
)

In [0]:
display(gold_monthly_sales)

In [0]:
# MoM growth
window_month = Window.orderBy("sales_month")

gold_monthly_sales = (
    gold_monthly_sales
    .withColumn("prior_month_sales", F.lag("total_net_sales").over(window_month))
    .withColumn(
        "mom_growth_pct",
        F.round(
            ((F.col("total_net_sales") - F.col("prior_month_sales")) / F.col("prior_month_sales")) * 100,
            2
        )
    )
)

In [0]:
display(gold_monthly_sales)

In [0]:
# Product performance 

gold_product_performance = (
    fact_prescriptions
    .groupBy("product_id", "product_name", "therapeutic_area")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_net_sales"),
        F.countDistinct("prescription_id").alias("total_prescriptions"),
        F.round(F.avg("net_sales"), 2).alias("avg_net_sales_per_rx")
    )
    .orderBy(F.desc("total_net_sales"))
)

In [0]:
display(gold_product_performance)

In [0]:
# Provider Performance 
gold_provider_performance = (
    fact_prescriptions
    .groupBy("provider_id", "provider_name", "specialty")
    .agg(
        F.countDistinct("prescription_id").alias("total_prescriptions"),
        F.round(F.sum("net_sales"), 2).alias("total_net_sales")
    )
    .orderBy(F.desc("total_prescriptions"))
)

In [0]:
display(gold_provider_performance)

In [0]:
# Region Performance

gold_region_performance = (
    fact_prescriptions
    .groupBy("region_id", "region_name", "sales_director")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_net_sales"),
        F.countDistinct("prescription_id").alias("total_prescriptions"),
        F.countDistinct("provider_id").alias("active_providers")
    )
    .orderBy(F.desc("total_net_sales"))
)

In [0]:
display(gold_region_performance)

In [0]:
# Payer mix 
gold_payer_mix = (
    fact_prescriptions
    .groupBy("payer_type")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_net_sales"),
        F.countDistinct("prescription_id").alias("total_prescriptions"),
        F.round(F.avg("discount"), 2).alias("avg_discount")
    )
    .orderBy(F.desc("total_net_sales"))
)

In [0]:
display(gold_payer_mix)